# 04 — Outline Generation

Turn one saved `VideoTopic` into a timed, structured outline for an
educational short video.

This notebook:

1. Loads a saved topic collection.
2. Selects one topic.
3. Generates a validated outline with the local LLM.
4. Saves the outline as JSON.
5. Previews the hook, sections, visuals, and timing.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.outlines import (
    build_outline_filename,
    find_topic,
    find_topic_file,
    generate_outline,
    save_outline,
)
from educational_shorts.prompts import load_prompt
from educational_shorts.topics import load_topics

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

In [2]:
TOPICS_DIRECTORY = PROJECT_ROOT / "data" / "topics"
OUTLINES_DIRECTORY = PROJECT_ROOT / "data" / "outlines"

# Set this to a specific JSON filename, or leave it as None to use the
# most recently modified topic file.
TOPICS_FILENAME = None

# Select by exact title. Set to None to use TOPIC_INDEX instead.
SELECTED_TOPIC_TITLE = "How Do Bacteria Communicate?"
TOPIC_INDEX = 0

TARGET_SECONDS = 60
SECTION_COUNT = 4
TEMPERATURE = 0.4
GENERATION_SEED = 42

print(f"Topics directory: {TOPICS_DIRECTORY}")
print(f"Outlines directory: {OUTLINES_DIRECTORY}")

Topics directory: c:\Users\hitch\python_files\educational_shorts\data\topics
Outlines directory: c:\Users\hitch\python_files\educational_shorts\data\outlines


## Load topics and select one

In [3]:
topics_path = find_topic_file(
    topics_directory=TOPICS_DIRECTORY,
    filename=TOPICS_FILENAME,
)

topic_collection = load_topics(topics_path)

selected_topic = find_topic(
    topic_list=topic_collection,
    title=SELECTED_TOPIC_TITLE,
    index=TOPIC_INDEX,
)

print(f"Loaded topics from: {topics_path}")
print(f"Selected topic: {selected_topic.title}")
print(f"Objective: {selected_topic.learning_objective}")

Loaded topics from: c:\Users\hitch\python_files\educational_shorts\data\topics\science__biology__microbiology.json
Selected topic: How Do Bacteria Communicate?
Objective: Understand how bacteria use chemical signals to communicate and coordinate behavior.


## Load the outline-generation prompt

In [4]:
outline_system_prompt = load_prompt("outline_generation")

print("Outline-generation prompt loaded.")

Outline-generation prompt loaded.


## Generate the outline

In [5]:
video_outline = generate_outline(
    topic=selected_topic,
    system_prompt=outline_system_prompt,
    target_seconds=TARGET_SECONDS,
    section_count=SECTION_COUNT,
    temperature=TEMPERATURE,
    seed=GENERATION_SEED,
)

print(
    f"Generated an outline with {len(video_outline.sections)} body sections."
)
print(
    f"Estimated total duration: "
    f"{video_outline.estimated_total_seconds} seconds"
)

Generated an outline with 4 body sections.
Estimated total duration: 60 seconds


## Save the outline

In [6]:
output_path = (
    OUTLINES_DIRECTORY
    / build_outline_filename(selected_topic)
)

save_outline(
    outline=video_outline,
    output_path=output_path,
)

print(f"Saved outline to {output_path}")

Saved outline to c:\Users\hitch\python_files\educational_shorts\data\outlines\how_do_bacteria_communicate.json


## Preview

In [7]:
print(f"TITLE: {video_outline.topic.title}")
print(f"HOOK: {video_outline.hook}")
print()

for index, section in enumerate(video_outline.sections, start=1):
    print(
        f"{index}. {section.section_type.upper()} "
        f"({section.estimated_seconds}s)"
    )
    print(f"   Purpose: {section.purpose}")

    for point in section.key_points:
        print(f"   - {point}")

    print(f"   Visual: {section.visual_direction}")
    print()

print(f"CLOSING TAKEAWAY: {video_outline.closing_takeaway}")
print(
    f"ESTIMATED TOTAL: {video_outline.estimated_total_seconds} seconds"
)

TITLE: How Do Bacteria Communicate?
HOOK: Did you know that bacteria can talk to each other? They don’t use words, but they do have a secret language made of chemicals.

1. INTRODUCTION TO BACTERIAL COMMUNICATION (15s)
   Purpose: Introduce the concept of bacterial communication and its importance.
   - Bacteria are not solitary; they interact with each other.
   - They use chemical signals called quorum sensing.
   Visual: Show a simple animation of bacteria in a colony, with glowing dots representing chemical signals.

2. WHAT IS QUORUM SENSING? (15s)
   Purpose: Explain the mechanism and purpose of quorum sensing.
   - Bacteria release molecules into their environment.
   - These molecules act as messages to other bacteria.
   Visual: Use a diagram showing bacteria releasing signal molecules, with arrows indicating communication pathways.

3. EXAMPLES OF BACTERIAL COMMUNICATION (15s)
   Purpose: Provide real-world examples of how this communication affects behavior.
   - Bacteria co